<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/06_secuencias/61_seq2seq_atencion.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Seq2Seq con atención

**Pregunta guía:** ¿Cómo aprende un decoder qué partes de la entrada consultar?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere PyTorch.** Un encoder produce estados $h_1,\ldots,h_T$. En el
paso $s$, atención calcula $\alpha_{st}=\mathrm{softmax}(q_s^Th_t)$ y
contexto $c_s=\sum_t\alpha_{st}h_t$. Entrenaremos una tarea transparente:
invertir secuencias de dígitos. Los mapas de atención tienen una respuesta
esperada: una diagonal invertida.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

SEMILLA=42; torch.manual_seed(SEMILLA); rng=np.random.default_rng(SEMILLA)
dispositivo=torch.device("cuda" if torch.cuda.is_available() else "cpu")
VOCAB=11; SOS=10; L=6; N=3000
fuente=torch.tensor(rng.integers(0,10,size=(N,L)),dtype=torch.long)
objetivo=torch.flip(fuente,dims=[1])
train,val,test=torch.arange(0,2200),torch.arange(2200,2600),torch.arange(2600,N)

class Seq2SeqAtención(nn.Module):
    def __init__(self,emb=24,oculto=48):
        super().__init__(); self.embed=nn.Embedding(VOCAB,emb); self.encoder=nn.GRU(emb,oculto,batch_first=True)
        self.decoder=nn.GRUCell(emb+oculto,oculto); self.salida=nn.Linear(2*oculto,10)
    def forward(self,x,target=None):
        enc,h=self.encoder(self.embed(x)); h=h.squeeze(0); previo=torch.full((len(x),),SOS,device=x.device)
        logits=[]; atenciones=[]
        for s in range(L):
            pesos=torch.softmax(torch.bmm(enc,h.unsqueeze(2)).squeeze(2),dim=1)
            contexto=torch.bmm(pesos.unsqueeze(1),enc).squeeze(1)
            h=self.decoder(torch.cat([self.embed(previo),contexto],dim=1),h)
            logit=self.salida(torch.cat([h,contexto],dim=1)); logits.append(logit); atenciones.append(pesos)
            previo=target[:,s] if target is not None else logit.argmax(1)
        return torch.stack(logits,dim=1),torch.stack(atenciones,dim=1)


In [ ]:
modelo=Seq2SeqAtención().to(dispositivo); opt=torch.optim.Adam(modelo.parameters(),lr=2e-3); ce=nn.CrossEntropyLoss()
for época in range(35):
    perm=train[torch.randperm(len(train))]
    modelo.train()
    for inicio in range(0,len(perm),128):
        idx=perm[inicio:inicio+128]; xb=fuente[idx].to(dispositivo); yb=objetivo[idx].to(dispositivo)
        opt.zero_grad(); logits,_=modelo(xb,yb); loss=ce(logits.reshape(-1,10),yb.reshape(-1)); loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(),1.0); opt.step()
    if época%10==0:
        modelo.eval()
        with torch.no_grad(): pred=modelo(fuente[val].to(dispositivo))[0].argmax(2).cpu()
        print(época,"secuencias exactas",(pred==objetivo[val]).all(1).float().mean().item())


In [ ]:
modelo.eval()
with torch.no_grad(): logits,A=modelo(fuente[test[:1]].to(dispositivo)); pred=logits.argmax(2).cpu()
print("entrada",fuente[test[0]].tolist(),"objetivo",objetivo[test[0]].tolist(),"pred",pred[0].tolist())
plt.imshow(A[0].cpu(),cmap="viridis",vmin=0,vmax=1); plt.xlabel("posición de entrada"); plt.ylabel("paso de salida"); plt.colorbar(label="atención"); plt.show()


**Ejercicios:** quite atención y use sólo el último estado; aumente la
longitud; reemplace teacher forcing total por probabilidad decreciente;
explique por qué un mapa de atención no es automáticamente una explicación
causal de la decisión.
